
# Financial Risk Assessment (Data Processing)

Columns:
- **Demographic**: Age, Gender, Education Level, Marital Status, Number of Dependents, Marital Status Change
- **Financial**: Income, Credit Score, Loan Amount, Debt-to-Income Ratio, Assets Value, Previous Defaults
- **Employment**: Employment Status, Years at Current Job, Payment History
- **Location**: City, State, Country
- **Purpose**: Loan Purpose
- **Target**: Risk Rating (Low, Medium, High)

Preprocessing:
- Imputed 2,250 missing values per 6 columns with medians (e.g., Income: 69,773).
- Encoded 6 categorical variables into 16 binary columns.
- Fixed 4,757 inconsistencies (e.g., Years at Current Job set to 0 for unemployed).
- No rows lost to outliers or duplicates.

## Purpose
The Financial Risk Assessment model is designed to predict a college student’s financial risk level—categorized as Low, Medium, or High—based on their demographic, financial, and behavioral data. This model serves as a core component of the project, enabling personalized financial advice and supporting students in achieving their financial goals. By identifying risk levels, the model informs recommendations (e.g., “Reduce discretionary spending” for high-risk students) and contributes to calculating an optimal monthly expense plan tailored to each user’s risk profile.

## Data Foundation
**Input Features**: The model uses 29 features, including:
- **Demographic**: Age, gender (encoded as Male, Non-binary), education level (High School, Master’s, PhD), marital status (Married, Single, Widowed), and number of dependents.
- **Financial**: Income, credit score, loan amount, debt-to-income ratio, assets value, and previous defaults.
- **Employment**: Employment status (Self-employed, Unemployed), years at current job (adjusted to 0 for unemployed), and payment history (Fair, Good, Poor).
- **Loan Details**: Loan purpose (Business, Home, Personal).

**Target Variable**: Risk Rating (Low, Medium, High), a categorical label reflecting financial stability and risk tolerance.

**Preprocessing**: The dataset is fully cleaned—missing values (15% per key column) imputed with medians (e.g., Income: 69,773), categorical variables one-hot encoded into binary columns, inconsistencies resolved (e.g., unemployed job years set to 0), and no outliers or duplicates removed due to the data’s inherent consistency.

In [254]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# Load the dataset
df_risk = pd.read_csv('Assets/Risk/financial_risk_assessment.csv')

# Step 2: Analyze missingness pattern
missing_cols = ['Income', 'Credit Score', 'Loan Amount', 'Assets Value', 
                'Number of Dependents', 'Previous Defaults']
missing_rows = df_risk[missing_cols].isnull()

# Impute with median (future-proofed)
for col in missing_cols:
    median_value = df_risk[col].median()
    df_risk[col] = df_risk[col].fillna(median_value) 


# List of categorical columns to encode
cat_columns = ['Gender', 'Education Level', 'Marital Status', 'Employment Status', 
               'Payment History', 'Loan Purpose']

# One-hot encode (drop_first=True to avoid multicollinearity)
df_risk = pd.get_dummies(df_risk, columns=cat_columns, drop_first=True)

# Convert boolean encoded columns to integers (0/1)
encoded_cols = [col for col in df_risk.columns if col.startswith(('Gender_', 'Education Level_', 
                                                                 'Marital Status_', 'Employment Status_', 
                                                                 'Payment History_', 'Loan Purpose_'))]
for col in encoded_cols:
    df_risk[col] = df_risk[col].astype(int)

# Ensure numerical columns are proper types
num_columns = ['Age', 'Income', 'Credit Score', 'Loan Amount', 'Years at Current Job', 
               'Debt-to-Income Ratio', 'Assets Value', 'Number of Dependents', 'Previous Defaults', 
               'Marital Status Change']
for col in num_columns:
    df_risk[col] = pd.to_numeric(df_risk[col], errors='coerce')

# Check for inconsistencies: Unemployed with Years at Current Job > 0
inconsistent = df_risk[(df_risk['Employment Status_Unemployed'] == 1) & 
                       (df_risk['Years at Current Job'] > 0)]

# Fix: Set Years at Current Job to 0 for Unemployed
df_risk.loc[df_risk['Employment Status_Unemployed'] == 1, 'Years at Current Job'] = 0

# Function to detect and remove outliers using IQR
def remove_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df_filtered = df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]
    return df_filtered

# Columns to check for outliers
outlier_cols = ['Income', 'Credit Score', 'Loan Amount', 'Assets Value', 
                'Debt-to-Income Ratio', 'Previous Defaults', 'Age', 
                'Years at Current Job', 'Number of Dependents', 'Marital Status Change']

# Apply outlier removal
df_clean = df_risk.copy()
for col in outlier_cols:
    df_clean = remove_outliers(df_clean, col)

# Add domain知識 constraint for Credit Score (300-850)
df_clean = df_clean[(df_clean['Credit Score'] >= 300) & (df_clean['Credit Score'] <= 850)]

# Remove duplicates
df_clean = df_risk.drop_duplicates()

# Save checkpoint
df_clean.to_csv('Assets/Risk/cleaned_dataset_financial_risk.csv', index=False)

# Monthly Expenses (Data Processing)

## Input Data
The input data consists of 13 features describing college students' demographic, financial, and lifestyle information.

### Demographic Data
- **Gender**: The gender of the student (e.g., Male, Female).
- **Age**: The age of the student.
- **Study_year**: The year of study (e.g., 1 for Freshman, 2 for Sophomore, etc.).
- **Living**: The living arrangement of the student (e.g., Home, Hostel).

### Financial Data
- **Scholarship**: Whether the student has a scholarship (Yes, No).
- **Part_time_job**: Whether the student has a part-time job (Yes, No).
- **Monthly_expenses_$**: The monthly expenses of the student in dollars (this is the target variable we want to predict).

### Lifestyle Data
- **Transporting**: The mode of transportation used by the student (e.g., Motorcycle, No).
- **Smoking**: Whether the student smokes (Yes, No).
- **Drinks**: Whether the student drinks (Yes, No).
- **Games_&_Hobbies**: Whether the student spends on games and hobbies (Yes, No).
- **Cosmetics_&_Self-care**: Whether the student spends on cosmetics and self-care (Yes, No).
- **Monthly_Subscription**: Whether the student has monthly subscriptions (Yes, No).

In [268]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# Load the dataset
df = pd.read_csv('Assets/ME/Expenses.csv')

# Display the first few rows
print(df.head())

#Check column names and data types
print(df.info())

#Identify missing values
print(df.isnull().sum())

# For numerical columns: fill missing values with the median
df['Study_year'] = df['Study_year'].fillna(df['Study_year'].median())
df['Monthly_expenses_$'] = df['Monthly_expenses_$'].fillna(df['Monthly_expenses_$'].median())

# For categorical columns: fill missing values with the mode
categorical_columns = ['Living', 'Transporting', 'Smoking', 'Drinks', 'Cosmetics_&_Self-care', 'Monthly_Subscription', 'Part_time_job']
for col in categorical_columns:
    df[col] = df[col].fillna(df[col].mode()[0])

#Convert Categorical Columns to Appropriate Types:
categorical_columns = ['Gender', 'Living', 'Scholarship', 'Part_time_job', 'Transporting', 'Smoking', 'Drinks', 'Games_&_Hobbies', 'Cosmetics_&_Self-care', 'Monthly_Subscription']
df[categorical_columns] = df[categorical_columns].astype('category')

#Living Arrangement Binary
df['Living_Home'] = df['Living'].apply(lambda x: 1 if x == 'Home' else 0)

# One-hot encode all categorical columns and ensure the result is integers (0/1)
df = pd.get_dummies(df, columns=['Gender', 'Living', 'Scholarship', 'Part_time_job', 'Transporting', 'Smoking', 'Drinks', 'Games_&_Hobbies', 'Cosmetics_&_Self-care', 'Monthly_Subscription'], drop_first=True, dtype=int)

# Normalize numerical features
scaler = MinMaxScaler()
df[['Age', 'Study_year', 'Monthly_expenses_$']] = scaler.fit_transform(df[['Age', 'Study_year', 'Monthly_expenses_$']])

# Convert Age to numeric if needed
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')

#Part-Time Job and Transportation
df['Part_Time_Transport'] = df['Part_time_job_Yes'] * df['Transporting_Motorcycle']

#Financial Health Indicator
df['Financial_Health'] = df['Monthly_expenses_$'] / df['Monthly_expenses_$'].max()  

df.to_csv('Assets/ME/CleanedExpenses.csv', index=False)

    Gender  Age  Study_year  Living Scholarship Part_time_job Transporting  \
0  Female    21         2.0    Home          No            No           No   
1    Male    25         3.0  Hostel          No           Yes   Motorcycle   
2    Male    23         2.0    Home         Yes            No           No   
3    Male    19         3.0  Hostel          No            No   Motorcycle   
4  Female    19         2.0    Home          No            No   Motorcycle   

  Smoking Drinks Games_&_Hobbies Cosmetics_&_Self-care Monthly_Subscription  \
0      No     No              No                   Yes                   No   
1      No     No             Yes                   Yes                  Yes   
2      No     No              No                    No                  NaN   
3      No     No             Yes                   Yes                  Yes   
4      No     No              No                   Yes                   No   

   Monthly_expenses_$  
0               150.0  
1       

# Earning and Loan (Data Processing)
The Earnings and Loan Repayment model utilizes the cleaned College Scorecard dataset (scorecard_step5_final.csv) to predict key financial outcomes for college students based on their institution’s historical data. Specifically, the model aims to forecast median post-graduation earnings (earnings_med) and potentially employment outcomes (derived from count_working and count_not_working). This model enhances the financial assistant project by providing students with personalized insights into their expected earnings and job prospects after graduation, supporting debt management, career planning, and financial goal-setting (e.g., saving for a laptop or paying off loans). By linking predictions to user-specified school names via inst_name, the model delivers tailored guidance integrated with the broader system’s risk and expense features.

# How It Works
The model is built on a cleaned dataset containing 25,686 rows and 64 columns, with earnings_med as the primary target variable and additional features like count_working and count_not_working offering employment context. Here’s how it operates:

Target Variable: earnings_med—median earnings of graduates (e.g., $36,600), a continuous value reflecting earning potential.

Preprocessing: The dataset is fully cleaned:

- Dropped 35.6% of rows (17,255) with missing values in critical columns (earnings_med, count_not_working, count_working).
- Removed 17.6% of remaining rows (5,504) as outliers (e.g., earnings_med > 61,400, count_working > 2,709).
- Encoded state_abbr into 57 binary columns, retained inst_name as text, ensured numeric consistency, and eliminated duplicates (none found).

In [269]:
import pandas as pd

# Load the dataset
df = pd.read_csv('Assets/Earning and Loan/scorecard.csv')

# Drop the unnamed index column
df = df.drop(columns=['Unnamed: 0'], errors='ignore')

# Define critical columns
missing_cols = ['pred_degree_awarded_ipeds', 'earnings_med', 'count_not_working', 'count_working']

# Check rows with NAs
missing_rows = df[missing_cols].isnull()

# Drop rows with any NA in critical columns
df_clean = df.dropna(subset=missing_cols, how='any')

# One-hot encode state_abbr (only categorical for modeling)
df_clean = pd.get_dummies(df_clean, columns=['state_abbr'], prefix='state', drop_first=True)

# Convert state columns (booleans) to integers
state_cols = [col for col in df_clean.columns if col.startswith('state_')]
for col in state_cols:
    df_clean[col] = df_clean[col].astype(int)

# Ensure numeric columns are proper types
num_cols = ['unitid', 'pred_degree_awarded_ipeds', 'year', 'earnings_med', 
            'count_not_working', 'count_working']
for col in num_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# Check for inconsistencies
# Negative values
negatives = df_clean[(df_clean['earnings_med'] < 0) | 
                    (df_clean['count_not_working'] < 0) | 
                    (df_clean['count_working'] < 0)]

# Year range
out_of_range = df_clean[(df_clean['year'] < 2000) | (df_clean['year'] > 2020)]

# pred_degree_awarded_ipeds valid values (e.g., 1, 2, 3)
invalid_degrees = df_clean[~df_clean['pred_degree_awarded_ipeds'].isin([1, 2, 3])]

# Drop rows with issues (if any)
df_clean = df_clean[(df_clean['earnings_med'] >= 0) & 
                    (df_clean['count_not_working'] >= 0) & 
                    (df_clean['count_working'] >= 0) & 
                    (df_clean['year'].between(2000, 2020)) & 
                    (df_clean['pred_degree_awarded_ipeds'].isin([1, 2, 3]))]

def remove_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    print(f"{column} - Lower bound: {lower_bound}, Upper bound: {upper_bound}")
    print(f"Rows outside bounds before: {df[(df[column] < lower_bound) | (df[column] > upper_bound)].shape[0]}")
    return df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]

# Check outliers in key columns
outlier_cols = ['earnings_med', 'count_not_working', 'count_working']
for col in outlier_cols:
    df_clean = remove_outliers(df_clean, col)

# Apply domain caps
df_clean = df_clean[(df_clean['earnings_med'] >= 0) & (df_clean['earnings_med'] <= 150000) &
                    (df_clean['count_not_working'] >= 0) & (df_clean['count_not_working'] <= 10000) &
                    (df_clean['count_working'] >= 0) & (df_clean['count_working'] <= 10000)]

# Check for duplicates
print("Number of duplicate rows:", df_clean.duplicated().sum())

# Remove duplicates
df_clean = df_clean.drop_duplicates()

# Check shape
print("\nShape after removing duplicates:", df_clean.shape)
print("\nFirst few rows:")
print(df_clean.head())

# Save checkpoint
df_clean.to_csv('Assets/Earning and Loan/CleanedData.csv', index=False)
print("Successfully saved")

earnings_med - Lower bound: 3000.0, Upper bound: 61400.0
Rows outside bounds before: 680
count_not_working - Lower bound: -339.0, Upper bound: 709.0
Rows outside bounds before: 2747
count_working - Lower bound: -1227.0, Upper bound: 2709.0
Rows outside bounds before: 2077
Number of duplicate rows: 0

Shape after removing duplicates: (25686, 64)

First few rows:
   unitid                            inst_name  pred_degree_awarded_ipeds  \
0  100654             Alabama A & M University                          3   
1  100663  University of Alabama at Birmingham                          3   
3  100706  University of Alabama in Huntsville                          3   
4  100724             Alabama State University                          3   
6  100760    Central Alabama Community College                          2   

   year  earnings_med  count_not_working  count_working  state_AL  state_AR  \
0  2007       36600.0              116.0         1139.0         1         0   
1  2007       4

# Avg Cost by State (Data Processing)

This section focuses on the steps required to process the data before analysis. The key steps include:

1. **Data Cleaning**: Handle missing values, remove duplicates, and correct inconsistencies in the dataset.
2. **Data Transformation**: Normalize, scale, or encode data as needed for analysis or modeling.
3. **Feature Engineering**: Create new features or modify existing ones to improve model performance.

**Target Variable**: Value—a continuous cost amount (e.g., $13,983 for tuition, $8,503 for room/board), representing the predicted expense.

The dataset is fully cleaned with the following steps:

- **Initial State**: Started with 3,548 rows, no missing values.
- **Year Filtering**: Dropped 345 rows (9.7%) with years outside 2000–2020 for relevance.
- **Outlier Removal**: Removed 388 outliers (12.1%) where `Value` exceeded ~24,801, keeping costs realistic.
- **Encoding**: Encoded `State`, `Type`, `Length`, and `Expense` into 54 binary columns, no duplicates found.


In [270]:
# Load the pandas library for data manipulation.
import pandas as pd


In [271]:
# Load the dataset
df_cost = pd.read_csv('Assets/Avg Cost by state/nces330_20.csv')
df_cost.head()

,Year,State,Type,Length,Expense,Value
0,2013,Alabama,Private,4-year,Fees/Tuition,13983
1,2013,Alabama,Private,4-year,Room/Board,8503
2,2013,Alabama,Public In-State,2-year,Fees/Tuition,4048
3,2013,Alabama,Public In-State,4-year,Fees/Tuition,8073
4,2013,Alabama,Public In-State,4-year,Room/Board,8473


### Encode Categorical Columns
Convert categorical variables (State, Type, Length, Expense) into dummy variables, dropping the first category to avoid multicollinearity.

In [272]:
cat_cols = ['State', 'Type', 'Length', 'Expense']
df_clean = pd.get_dummies(df_cost, columns=cat_cols, prefix=['state', 'type', 'length', 'expense'], drop_first=True)
df_clean.head()  # Check the result

,Year,Value,state_Alaska,state_Arizona,state_Arkansas,state_California,state_Colorado,state_Connecticut,state_Delaware,state_District of Columbia,...,state_Vermont,state_Virginia,state_Washington,state_West Virginia,state_Wisconsin,state_Wyoming,type_Public In-State,type_Public Out-of-State,length_4-year,expense_Room/Board
0,2013,13983,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
1,2013,8503,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,True
2,2013,4048,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False
3,2013,8073,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,True,False,True,False
4,2013,8473,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,True,False,True,True


### Convert Booleans to Integers
Ensure dummy variables are integers (0/1) instead of booleans for consistency.

In [273]:
bool_cols = [col for col in df_clean.columns if col.startswith(('state_', 'type_', 'length_', 'expense_'))]
for col in bool_cols:
    df_clean[col] = df_clean[col].astype(int)

### Ensure Numeric Columns
Convert Year and Value to numeric types, handling any non-numeric entries gracefully.

In [274]:
df_clean['Year'] = pd.to_numeric(df_clean['Year'], errors='coerce')
df_clean['Value'] = pd.to_numeric(df_clean['Value'], errors='coerce')
df_clean.dtypes  #Verify column types

Year                          int64
Value                         int64
state_Alaska                  int64
state_Arizona                 int64
state_Arkansas                int64
state_California              int64
state_Colorado                int64
state_Connecticut             int64
state_Delaware                int64
state_District of Columbia    int64
state_Florida                 int64
state_Georgia                 int64
state_Hawaii                  int64
state_Idaho                   int64
state_Illinois                int64
state_Indiana                 int64
state_Iowa                    int64
state_Kansas                  int64
state_Kentucky                int64
state_Louisiana               int64
state_Maine                   int64
state_Maryland                int64
state_Massachusetts           int64
state_Michigan                int64
state_Minnesota               int64
state_Mississippi             int64
state_Missouri                int64
state_Montana               

### Check Data Consistency
Identify negative values in 'Value' and years outside the 2000-2020 range.

In [275]:
negatives = df_clean[df_clean['Value'] < 0] # type: ignore
out_of_range = df_clean[(df_clean['Year'] < 2000) | (df_clean['Year'] > 2020)] # type: ignore
print(f"Negative Values: {negatives.shape[0]} rows")
print(f"Out-of-Range Years: {out_of_range.shape[0]} rows")

Negative Values: 0 rows
Out-of-Range Years: 345 rows


### Drop Invalid Rows
Remove rows with negative values or years outside 2000-2020.

In [276]:
df_clean = df_clean[(df_clean['Value'] >= 0) & 
                    (df_clean['Year'].between(2000, 2020))]
df_clean.shape  # Check new size

(3203, 56)

### Define Outlier Removal Function
Create a function to remove outliers from a column using the IQR method.

In [277]:
def remove_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    print(f"{column} - Lower: {lower}, Upper: {upper}")
    print(f"Rows outside:", df[(df[column] < lower) | (df[column] > upper)].shape[0])
    return df[(df[column] >= lower) & (df[column] <= upper)]

### Remove Outliers from Value
Apply the outlier removal function to the 'Value' column.

In [278]:
df_clean = remove_outliers(df_clean, 'Value')

Value - Lower: -2576.75, Upper: 24801.25
Rows outside: 388


### Apply Domain-Specific Cap
Restrict 'Value' to a reasonable range (0 to 75,000) based on domain knowledge.

In [279]:
df_clean = df_clean[(df_clean['Value'] >= 0) & (df_clean['Value'] <= 75000)] # type: ignore
df_clean.describe()  # Summary stats

,Year,Value,state_Alaska,state_Arizona,state_Arkansas,state_California,state_Colorado,state_Connecticut,state_Delaware,state_District of Columbia,...,state_Vermont,state_Virginia,state_Washington,state_West Virginia,state_Wisconsin,state_Wyoming,type_Public In-State,type_Public Out-of-State,length_4-year,expense_Room/Board
count,2815.000000,2815.000000,2815.000000,2815.000000,2815.000000,2815.000000,2815.000000,2815.000000,2815.000000,2815.000000,...,2815.000000,2815.000000,2815.000000,2815.000000,2815.000000,2815.000000,2815.000000,2815.000000,2815.000000,2815.000000
mean,2016.417407,10227.934636,0.020249,0.021314,0.022735,0.017052,0.019893,0.017052,0.017052,0.008526,...,0.017052,0.019893,0.017052,0.022735,0.019183,0.018828,0.425933,0.365542,0.722202,0.426288
std,2.282512,4959.059654,0.140875,0.144456,0.149085,0.129486,0.139659,0.129486,0.129486,0.091957,...,0.129486,0.139659,0.129486,0.149085,0.137192,0.135940,0.494571,0.481667,0.447993,0.494625
min,2013.000000,1225.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2014.000000,7284.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2016.000000,9506.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
75%,2018.000000,12125.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000
max,2020.000000,24791.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


### Remove Duplicates
Drop any duplicate rows to ensure data integrity.

In [280]:
df_clean = df_clean.drop_duplicates()
df_clean.shape  # Final size check

(2815, 56)

### Save the Cleaned Dataset
Export the processed data to a new CSV file for later use (e.g., model training).

In [281]:
df_clean.to_csv('Assets/Avg Cost by state/cleaned.csv', index=False)
print("Data saved successfully!")

Data saved successfully!
